In [ ]:
from pathlib import Path
import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d01_init_proc import align_imgs
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn

import numpy as np
import pandas as pd
import random

In [ ]:
input_dirpath = Path(input())

In [ ]:
imgnames = [p.name for p in input_dirpath.glob('*.ome.tif')]
imgnames.sort()
df = pd.DataFrame({'img name': imgnames})
df.head()

In [ ]:
namesplits = df['img name'].str.replace('.ome.tif', '').str.split('_')
namesplits[0]

In [ ]:
df['exp'] = namesplits.str[0]
df['DIV'] = namesplits.str[1]
df['scene'] = namesplits.str[3]
df['wellID'] = df['exp'] + '-' + namesplits.str[2]
df.head()

In [ ]:
seed_value = 0
random.seed(seed_value)
rand_int = np.arange(0, len(df))
random.shuffle(rand_int)
df['blinded name'] = df['exp'] + '_' + pd.Series(rand_int).astype(str) + '.ome.tif'

assert len(df)==len(df['blinded name'].unique())
df.head()

In [ ]:
proc_dirpath = utils.get_proc_dirpath(input_dirpath)
tables_dirpath = proc_dirpath / dn.tables_dirname
tables_dirpath.mkdir(exist_ok=True)

key_df_path = tables_dirpath / 'blinding_key.csv'
df.to_csv(key_df_path)

In [ ]:
key_df_path = Path(input())

In [ ]:
df = pd.read_csv(key_df_path)
df.head()

In [ ]:
# apply blinding

for i, row in df.iterrows():
    imgpath = input_dirpath / row['img name']
    if imgpath.is_file():
        imgpath.rename(input_dirpath / row['blinded name'])

In [ ]:
input_dirpath = Path(input())

In [ ]:
# reverse blinding for images

# load key for blinded image names
df = pd.read_csv(key_df_path)
df['blinded basename'] = df['blinded name'].str.split('.ome.tif').str[0]
df


imgnames_ext = [p.name for p in input_dirpath.glob('*')]

df_ext = pd.DataFrame({'blinded name ext': imgnames_ext})
df_ext['blinded basename'] = df_ext['blinded name ext'].str.split('_ROI').str[0]
df_ext

print(df_ext)

prev_df_len = len(df_ext)
df = pd.merge(df_ext, df)

assert len(df)==prev_df_len

df['img basename'] = df['img name'].str.split('.ome.tif').str[0]
#df['unblinded name ext'] = df['blinded name ext'].str.replace(df['blinded basename'], df['img basename'])

df['unblinded name ext'] = df.apply(lambda x: x['blinded name ext'].replace(x['blinded basename'], str(x['img basename'])), axis=1)
assert len(df['unblinded name ext'].unique()) == len(df)

df

In [ ]:
for i, row in df.iterrows():

    imgpath = input_dirpath / row['blinded name ext']
    if imgpath.is_file():
        imgpath.rename(input_dirpath / row['unblinded name ext'])